In [1]:
import dspy

turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=250)
dspy.settings.configure(lm=turbo)

/home/ash/miniconda3/envs/nanites/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dspy.datasets.gsm8k import GSM8K, gsm8k_metric

gms8k = GSM8K()

trainset, devset = gms8k.train, gms8k.dev

100%|██████████| 1319/1319 [00:00<00:00, 60951.09it/s]


In [3]:
print(trainset)

[Example({'question': "The result from the 40-item Statistics exam Marion and Ella took already came out. Ella got 4 incorrect answers while Marion got 6 more than half the score of Ella. What is Marion's score?", 'gold_reasoning': "Ella's score is 40 items - 4 items = <<40-4=36>>36 items. Half of Ella's score is 36 items / 2 = <<36/2=18>>18 items. So, Marion's score is 18 items + 6 items = <<18+6=24>>24 items.", 'answer': '24'}) (input_keys={'question'}), Example({'question': "Stephen made 10 round trips up and down a 40,000 foot tall mountain. If he reached 3/4 of the mountain's height on each of his trips, calculate the total distance he covered.", 'gold_reasoning': 'Up a mountain, Stephen covered 3/4*40000 = <<3/4*40000=30000>>30000 feet. Coming down, Stephen covered another 30000 feet, making the total distance covered in one round to be 30000+30000 = <<30000+30000=60000>>60000. Since Stephen made 10 round trips up and down the mountain, he covered 10*60000 = <<10*60000=600000>>60

In [4]:
class CoT(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought("question -> answer")

    def forward(self, question):
        return self.prog(question=question)

In [5]:
from dspy.evaluate import Evaluate

evaluate = Evaluate(devset=devset[:], metric=gsm8k_metric, num_threads=8, display_progress=True, display_table=False)

In [6]:
program = CoT()

evaluate(program, devset=devset[:])

 		You are using the client GPT3, which will be removed in DSPy 2.6.
 		Changing the client is straightforward and will let you use new features (Adapters) that improve the consistency of LM outputs, especially when using chat LMs. 

 		Learn more about the changes and how to migrate at
 		https://github.com/stanfordnlp/dspy/blob/main/examples/migration.ipynb
 		You are using the client GPT3, which will be removed in DSPy 2.6.
 		Changing the client is straightforward and will let you use new features (Adapters) that improve the consistency of LM outputs, especially when using chat LMs. 

 		Learn more about the changes and how to migrate at
 		https://github.com/stanfordnlp/dspy/blob/main/examples/migration.ipynb


Average Metric: 189.00 / 300 (63.0%): 100%|██████████| 300/300 [00:54<00:00,  5.46it/s]

2025/01/15 13:49:44 INFO dspy.evaluate.evaluate: Average Metric: 189 / 300 (63.0%)


63.0

In [7]:
# Import the optimizer
from dspy.teleprompt import MIPROv2

# Initialize optimizer
teleprompter = MIPROv2(
    metric=gsm8k_metric,
    auto="light", # Can choose between light, medium, and heavy optimization runs
)

# Optimize program
print(f"Optimizing program with MIPRO...")
optimized_program = teleprompter.compile(
    program.deepcopy(),
    trainset=trainset,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    requires_permission_to_run=False,
)

# Save optimize program for future use
optimized_program.save(f"mipro_optimized")

# Evaluate optimized program
print(f"Evaluate optimized program...")
evaluate(optimized_program, devset=devset[:])

2025/01/15 14:08:17 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 5
valset size: 100

2025/01/15 14:08:17 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/01/15 14:08:17 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/01/15 14:08:17 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Optimizing program with MIPRO...
Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


 10%|█         | 4/40 [00:04<00:43,  1.20s/it]


Bootstrapped 3 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/5


 15%|█▌        | 6/40 [00:07<00:40,  1.19s/it]


...


Average Metric: 16.00 / 25 (64.0%): 100%|██████████| 25/25 [00:06<00:00,  3.76it/s]

2025/01/15 14:10:06 INFO dspy.evaluate.evaluate: Average Metric: 16 / 25 (64.0%)
2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 64.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 4', 'Predictor 0: Few-Shot Set 4'].
2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [64.0, 52.0, 64.0, 52.0, 76.0, 68.0, 64.0]
2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [55.0]
2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 55.0
2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2025/01/15 14:10:06 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 76.0) from minibatch trials...



Average Metric: 69.00 / 100 (69.0%): 100%|██████████| 100/100 [00:18<00:00,  5.28it/s]

2025/01/15 14:10:25 INFO dspy.evaluate.evaluate: Average Metric: 69 / 100 (69.0%)
2025/01/15 14:10:25 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 69.0
2025/01/15 14:10:25 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [55.0, 69.0]
2025/01/15 14:10:25 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 69.0
2025/01/15 14:10:25 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/01/15 14:10:25 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/01/15 14:10:25 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 69.0!


ValueError: `path` must end with `.json` or `.pkl` when `save_program=False`, but received: mipro_optimized